In [2]:
import os
import pandas as pd

# Chargement robuste du fichier nettoyé
chemin = (
    "data/titanic_propre.csv"
    if os.path.exists("data/titanic_propre.csv")
    else "../data/titanic_propre.csv"
)
titanic = pd.read_csv(chemin)
nb_avant = len(titanic)
print(f"Lignes initiales : {nb_avant}")

# 1. Création de la table de correspondance des ports
ports = pd.DataFrame(
    {
        "embark_town": ["Southampton", "Cherbourg", "Queenstown"],
        "pays": ["Angleterre", "France", "Irlande"],
    }
)
print("\n--- Table référentielle des ports ---")
print(ports)

# 2. Jointure à gauche (left join) et contrôle d'intégrité
titanic_enrichi = titanic.merge(
    ports, on="embark_town", how="left", validate="many_to_one"
)
assert (
    len(titanic_enrichi) == nb_avant
), "Erreur : le nombre de lignes a été altéré !"
print(f"\nLignes après jointure réussie : {len(titanic_enrichi)}")

# 3. Effectifs et taux de survie par pays
stats_pays = titanic_enrichi.groupby("pays")["survived"].agg(
    passagers="count", taux_survie=lambda x: round(x.mean() * 100, 1)
)
print("\n--- Répartition des passagers et taux de survie par pays ---")
print(stats_pays)

# 4. Expérimentation : Retirer Queenstown de la table ports
ports_sans_q = ports[ports["embark_town"] != "Queenstown"]

# Test A : Jointure 'inner'
df_inner = titanic.merge(ports_sans_q, on="embark_town", how="inner")
lignes_perdues = len(titanic) - len(df_inner)
print(f"\nJointure 'inner' sans Queenstown : {len(df_inner)} lignes ({lignes_perdues} lignes perdues)")

# Test B : Jointure 'left'
df_left = titanic.merge(ports_sans_q, on="embark_town", how="left")
nan_generes = df_left["pays"].isna().sum()
print(f"Jointure 'left' sans Queenstown  : {len(df_left)} lignes ({nan_generes} valeurs NaN dans 'pays')")


Lignes initiales : 889

--- Table référentielle des ports ---
   embark_town        pays
0  Southampton  Angleterre
1    Cherbourg      France
2   Queenstown     Irlande

Lignes après jointure réussie : 889

--- Répartition des passagers et taux de survie par pays ---
            passagers  taux_survie
pays                              
Angleterre        644         33.7
France            168         55.4
Irlande            77         39.0

Jointure 'inner' sans Queenstown : 812 lignes (77 lignes perdues)
Jointure 'left' sans Queenstown  : 889 lignes (77 valeurs NaN dans 'pays')
